In [2]:
import altair as alt
import numpy as np
import pandas as pd

from ecostyles import EcoStyles
styles = EcoStyles()        # Create styles instance
styles.register_and_enable_theme(theme_name="article")      # Register and enable theme. "article" or "cotd"

In [3]:
styles.eco_colours

{'red': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4'}

#### Figure 1. Exchange rate

In [5]:
# Read csv data into pandas dataframe
data1 = pd.read_csv('data/Colombia article data(Sheet1).csv')
data1

,Date,TRM
0,2025/06/01,"4,148.72"
1,2025/06/02,"4,148.72"
2,2025/06/03,"4,148.72"
3,2025/06/04,"4,136.71"
4,2025/06/05,"4,107.85"
...,...,...
411,2026/07/17,"3,221.41"
412,2026/07/18,"3,262.58"
413,2026/07/19,"3,262.58"
414,2026/07/20,"3,262.58"


- Convert date format to yyyy-mm-dd
- Convert values from string type to numerical (float) 

In [7]:
# Convert to standard iso format (replace '/' with '-')
data1['Date'] = data1['Date'].str.replace('/', '-')

# Remove non-numeric characters from 'TRM' column and convert to numeric
data1['TRM'] = data1['TRM'].str.replace(r'[^\d.]', '', regex=True).astype(float)    # i.e. regex expression for all non-numeric characters except for the decimal point
# data1['TRM'] = data1['TRM'].str.replace(',', '').astype(float)     # In this case, equivalent to just removing the ','

data1

,Date,TRM
0,2025-06-01,4148.72
1,2025-06-02,4148.72
2,2025-06-03,4148.72
3,2025-06-04,4136.71
4,2025-06-05,4107.85
...,...,...
411,2026-07-17,3221.41
412,2026-07-18,3262.58
413,2026-07-19,3262.58
414,2026-07-20,3262.58


In [54]:
data1['TRM'].iloc[-1]

np.float64(3262.58)

In [100]:
# Calculate change from latest value for each row
data1['change_to_latest'] = data1['TRM'].apply(lambda x: (data1['TRM'].iloc[-1] - x) / x * 100)

# Format as %
data1['change_to_latest'] /= 100
data1['change_to_latest'] = data1['change_to_latest'].round(3)
import os
os.makedirs('data', exist_ok=True)
data1.to_csv('data/fig1_copusd_daily.csv', index=False)

data1

,Date,TRM,change_to_latest
0,2025-06-01,4148.72,-0.214
1,2025-06-02,4148.72,-0.214
2,2025-06-03,4148.72,-0.214
3,2025-06-04,4136.71,-0.211
4,2025-06-05,4107.85,-0.206
...,...,...,...
411,2026-07-17,3221.41,0.013
412,2026-07-18,3262.58,0.000
413,2026-07-19,3262.58,0.000
414,2026-07-20,3262.58,0.000


In [98]:
line = alt.Chart(data1).mark_line().encode(
    x=alt.X('Date:T').scale(domain=[data1['Date'].min(), '2026-08-01']),
    y=alt.Y('TRM:Q').title('Peso per US$, daily average').scale(
        zero=False, padding=40
    ),
)

label = alt.Chart(data1).transform_calculate(
    text_label="[timeFormat(datum.Date,'%d %b %Y'),format(datum.TRM, ',d') + ' Peso per US$']"
).mark_text(dy=-13, dx=8).encode(
    x=alt.X("Date:T").aggregate('max'),
    y=alt.Y("TRM:Q").aggregate({"argmax": "Date"}),
    text=alt.Text("text_label:N").aggregate({"argmax": "Date"})
)

point = alt.Chart(data1).mark_point().encode(
    x=alt.X("Date:T").aggregate('max'),
    y=alt.Y("TRM:Q").aggregate({"argmax": "Date"}),
)

chart1 = line + label + point
chart1.display()

styles.save(chart1, 'charts', 'fig1_copusd', width=420, height=280)

alt.LayerChart(...)

, "month(datum.Date) == 7", "day(datum.Date) == 21"

In [97]:

data1[data1['Date'].isin(['2025-07-21', '2026-07-21'])]

,Date,TRM,change_to_latest
50,2025-07-21,4000.61,-0.184479
415,2026-07-21,3262.58,0.000000


In [87]:
# Rule 1 year ago
base_rule = alt.Chart(data1).encode(
    x=alt.X('Date:T')
).transform_filter("year(datum.Date) == 2025").transform_filter("month(datum.Date) == 6").transform_filter("date(datum.Date) == 21")
# Filter to date a year before latest data 


rule = base_rule.mark_rule(color='gray', strokeDash=[5, 5])

rule_text = base_rule.mark_text(fontSize=12, yOffset=-15).encode(
    y=alt.value('height'),
    text=alt.Text('change_to_latest:Q').format('.1%')
)

rule + rule_text + chart1

alt.LayerChart(...)

<br>
<br>
<br>

#### Figure 2. Fiscal deficit

In [ ]:
# Read csv data into pandas dataframe 
data2 = pd.read_csv('data/Colombia article data(Sheet2).csv')
data2

,Date,Deficit
0,2006,-1.0
1,2007,-0.8
2,2008,0.0
3,2009,-2.7
4,2010,-3.3
5,2011,-2.0
6,2012,0.2
7,2013,-1.0
8,2014,-1.7
9,2015,-3.5


In [8]:
# Convert to yyyy-mm-dd format in string type (for altair charting)
data2['Date'] = data2['Date'].astype(str) + '-01-01'

In [94]:
line = alt.Chart(data2).mark_line().encode(
    x=alt.X("Date:T").scale(domain=['2006-01-01', '2026-05-01']).axis(
        domain=False,
        tickSize=5
    ),
    y=alt.Y("Deficit:Q").title('Government fiscal deficit, % of GDP').axis(
        gridDash = alt.expr("datum.value == 0 ? [1,0] : [1,5]"),
        gridWidth = alt.expr("datum.value == 0 ? 1.4 : 1"),
        labelExpr = "datum.value + '%'"
    )
)


label = alt.Chart(data2).transform_calculate(label="[year(datum.Date), datum.Deficit + '%']").mark_text(
    dy=-13, dx=8
).encode(
    x=alt.X("Date:T").aggregate('max'),
    y=alt.Y("Deficit:Q").aggregate({"argmax": "Date"}),
    text=alt.Text("label:N").aggregate({"argmax": "Date"})
)

point = alt.Chart(data2).mark_point().encode(
    x=alt.X("Date:T").aggregate('max'),
    y=alt.Y("Deficit:Q").aggregate({"argmax": "Date"}),
)

chart2 = line + label + point

styles.save(chart2, 'charts', 'fig2_deficit', width=420, height=280)
chart2.display()

alt.LayerChart(...)